PY ASSESMENT SOLUTION NOTEBOOK

In [44]:
# Numerical arrays
import numpy as np

#Math
import math

#C-like structs
import struct

# sys: checking the machine's native byte order (Problem 1F)
import sys

#For refernecingn
import hashlib

# Problem 1: Representing SHA-256 Data

SHA-256 is a cryptographic hash function that takes an input message of any length and produces a fixed size 256 bits (32 bytes) output.

## Sub-section A: 32-bit words

32-bit word is a number represented using 32 binary digits. It can hold a value from 0 to 2³² − 1
A word is a w-bit string that may be represented as a sequence of hex digits.

The 32-bit words can be displayed in binary, hex and decimal. To convert a word to hex digits, each 4-bit string is converted to its hex digit equivalent.

In Python, a 32-bit word can be represented using an integer (int). Python supports bitwise operations, such as AND, XOR and bit shifting, which are required by SHA-256. A mask can be used to keep the value within 32 bits.

In [45]:
word = 10
mask = 0xFFFFFFFF

print(bin(word))           # 0b1010
print(hex(word))           # 0xa
print(word)                # 10

word = (word + 5) & mask
print(word)                # 15

0b1010
0xa
10
15


## Sub-section B: Sequences of 32-bit words

Sequences of 32-bit words can be represented using a NumPy array. A 512-bit message block contains 16 words, with each word containing 32 bits (16 × 32 = 512). These words are intermediate data used during SHA-256 processing.

The np.uint32 data type is used to represent unsigned 32-bit integers, ensuring that each word has a fixed size.

In [46]:
words = np.array(
    [0x61626380, 0, 0, 0x00000018],
    dtype=np.uint32
)

print(words)
print(words.dtype)
print(len(words))

[1633837952          0          0         24]
uint32
4


A list is suitable because SHA-256 processes words in a specific order. The dtype property confirms the data type, while len(words) returns the number of words in the array.

## Sub-Section C: Representing input messages

An input message can be represented as a Python string (str) when the input is textual. Before processing, the string must be encoded into a byte sequence because SHA-256 operates on the binary representation of the message rather than directly on Python characters.

Python's encode() method converts text into bytes using a specified character encoding. UTF-8 is a suitable choice for text because it supports a wide range of characters.

In [47]:
message = "abc"
data = message.encode("utf-8")

print(data)       # b'abc'
print(len(data))  # 3

b'abc'
3


The resulting bytes object contains three bytes, corresponding to the ASCII-compatible UTF-8 encodings of a, b and c.

Using bytes is appropriate because it provides an immutable sequence of values in the range 0–255. It also allows the message to be sliced into blocks and processed at the byte level. For non-textual input, the original data can be supplied directly as bytes without first converting it into a string.

The distinction between text and bytes is important because different encodings of the same visible text can produce different byte sequences and therefore different hashes.

## Sub-Section D: Representing 512-bit message blocks

SHA-256 processes the padded message in blocks of 512 bits, equivalent to 64 bytes. The message is padded by appending a single 1 bit, followed by enough 0 bits, and then a 64-bit representation of the original message length. The resulting message length must be a multiple of 512 bits

A Python bytes object is suitable for representing a block because each byte contains eight bits and a complete block can be represented by exactly 64 bytes.

In [48]:
block = bytes(64)

print(len(block))       # 64
print(len(block) * 8)   # 512

64
512


This creates a 64-byte block containing zeros, demonstrating the size of the representation. It is not, by itself, a correctly padded SHA-256 block.

## Sub-Section E: Representing the final 256-bit hash value

SHA-256 produces a fixed-length message digest of 256 bits, regardless of the input message length. This corresponds to 32 bytes, or 64 hexadecimal digits when displayed as text . 

The digest can be represented as a bytes object for binary processing or as a hexadecimal string for presentation.

In [49]:
digest = bytes(32)
hex_digest = digest.hex()

print(len(digest))      # 32
print(len(hex_digest))  # 64

32
64


This example uses zero bytes to demonstrate the output representation; it is not an actual SHA-256 digest.

For verification, Python's hashlib module can produce a reference digest:

In [50]:
reference = hashlib.sha256(b"abc").digest()

print(len(reference))  # 32
print(reference.hex())

32
ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad


In an implementation assignment, hashlib is used only as a reference for testing; the implementation in Problem 6 computes the digest independently. The custom implementation should calculate the digest itself.

Using bytes preserves the exact binary output, while hexadecimal encoding provides a readable representation that can be compared against known test vectors. This supports testing the implementation against published SHA-256 examples.

## Sub-Section F: Big-endian integers

Endianness is the order in which the bytes of a multi-byte integer are stored.
**Big-endian** puts the most significant byte first; **little-endian** puts it last.
FIPS 180-4 (§3.1) specifies big-endian throughout SHA-256: when parsing message bytes
into words, when writing the 64-bit length during padding, and when converting the
final hash words to bytes.

This matters in practice because most modern processors (x86, and ARM in its usual
configuration) are little-endian. Any conversion that relies on the machine's
native order will silently produce wrong words, so every conversion in this
notebook states the byte order explicitly.

In [51]:
print(sys.byteorder)  # 'little' on most machines

n = 0x61626380
print(n.to_bytes(4, "big").hex())     # 61626380: most significant byte first
print(n.to_bytes(4, "little").hex())  # 80636261: reversed

# Reading the same 4 bytes with the wrong order gives a different word
raw = b"abc\x80"
print(hex(int.from_bytes(raw, "big")))     # 0x61626380 (correct for SHA-256)
print(hex(int.from_bytes(raw, "little")))  # 0x80636261 (wrong)

# The same choice in struct and NumPy: '>' means big-endian
print(struct.pack(">I", n).hex())    # 32-bit word
print(struct.pack(">Q", 24).hex())   # 64-bit length field: 0000000000000018

assert np.frombuffer(raw, dtype=">u4")[0] == n
assert np.frombuffer(raw, dtype="<u4")[0] != n

little
61626380
80636261
0x61626380
0x80636261
61626380
0000000000000018


np.frombuffer(raw, dtype=np.uint32) uses native order, so on a little-endian machine it silently gives the wrong word. This is why ">u4" is always specified.

# Problem 2: SHA-256 Bitwise Operations

Two basic operations from FIPS 180-4, Section 3.2, are needed:

- **SHR<sup>n</sup>(x)**, shift right: `x >> n`. Bits pushed off the right end
  are lost, and zeros come in on the left.
- **ROTR<sup>n</sup>(x)**, rotate right: `(x >> n) | (x << (32 - n))`. Bits pushed
  off the right end come back in on the left, so no information is lost.

In [52]:
def rotr(x, n):
    """Rotate a 32-bit unsigned integer right by n bits (ROTR^n in FIPS 180-4).

    Args:
        x: The value to rotate (converted to np.uint32).
        n: Number of bit positions to rotate, 0 < n < 32.

    Returns:
        np.uint32: x rotated right by n bits.
    """
    x = np.uint32(x)
    n = np.uint32(n)
    # Right part: bits that move right. Left part: bits that wrapped around.
    return np.uint32((x >> n) | (x << (np.uint32(32) - n)))


# Tests
assert rotr(0x00000001, 1) == 0x80000000  # lowest bit wraps to the top
assert rotr(0x12345678, 4) == 0x81234567  # one hex digit = 4 bits
assert isinstance(rotr(1, 1), np.uint32)
print("rotr: all tests passed")

rotr: all tests passed


What’s going on: Take 0x12345678 and rotate it by 4 bits. One hex digit is 4 bits, so the last digit 8 moves to the front: 0x81234567. x << (32-n) would go past 32 bits in normal Python, but uint32 throws away anything above bit 31, so only the wrapped bits are left. The | joins the two halves.

The shift amount is also wrapped in np.uint32 so the result stays uint32 on every NumPy version. Mixing it with a plain Python int can turn the result into int64 on older versions.

## 1. Ch(x, y, z): Choose

$$Ch(x, y, z) = (x \land y) \oplus (\lnot x \land z)$$

For each bit position, `x` acts as a selector:
- if the bit of `x` is **1**, the output takes the bit of **y**
- if the bit of `x` is **0**, the output takes the bit of **z**

It works like an `if x then y else z` applied separately to each of the 32 bits.
`x & y` keeps y's bits where x is 1, and `~x & z` keeps z's bits where x is 0.
The two never overlap, so XOR combines them.

In SHA-256, Ch is applied to the working variables e, f and g in every round.

In [53]:
def Ch(x, y, z):
    """Choose function from FIPS 180-4 (Equation 4.2).

    For each bit, select the bit of y where x is 1, and the bit of z where x is 0.

    Args:
        x, y, z: 32-bit words (converted to np.uint32).

    Returns:
        np.uint32: (x AND y) XOR (NOT x AND z).
    """
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    return np.uint32((x & y) ^ (~x & z))


# Tests
# Single-bit truth table: the result should be y when x=1 and z when x=0.
for a in (0, 1):
    for b in (0, 1):
        for c in (0, 1):
            assert Ch(a, b, c) == (b if a else c)

# x all 1s means take all of y. x all 0s means take all of z.
assert Ch(0xFFFFFFFF, 0xAAAAAAAA, 0x55555555) == 0xAAAAAAAA
assert Ch(0x00000000, 0xAAAAAAAA, 0x55555555) == 0x55555555

# Mixed selector: top half from y, bottom half from z.
assert Ch(0xFFFF0000, 0x12345678, 0x9ABCDEF0) == 0x1234DEF0
print("Ch: all tests passed")

Ch: all tests passed


What’s going on: The last test shows the idea best. x = 0xFFFF0000 is 1s in the top half and 0s in the bottom half. So the top half comes from y (1234) and the bottom half comes from z (DEF0). ~x flips every bit, and on a uint32 that gives exactly 32 flipped bits. A plain Python int would give a negative number instead, which is a big reason to use NumPy here.

## 2. Maj(x, y, z): Majority